<a href="https://www.kaggle.com/code/murtazaabdullah2010/neoai-2026-day-2-audio-0-980?scriptVersionId=330117290" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import kagglehub
import librosa
from librosa.effects import time_stretch
from librosa import resample
from librosa.feature import  zero_crossing_rate, spectral_bandwidth, spectral_centroid

In [ ]:
import librosa
sample_sub= pd.read_csv("/kaggle/input/competitions/neoai-2026-day-2-audio/dataset/submission.csv")
train_ds = pd.read_csv("/kaggle/input/competitions/neoai-2026-day-2-audio/dataset/train.csv")
test_ds= pd.read_csv("/kaggle/input/competitions/neoai-2026-day-2-audio/dataset/test.csv")
path = "/kaggle/input/competitions/neoai-2026-day-2-audio/dataset"

In [ ]:
rates = [0.9, 0.95, 1.05, 1.1]
pitch_steps = [-1, 1]

In [ ]:
def extract_features(x, sr):
    zcr = librosa.feature.zero_crossing_rate(x)[0]
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13)
    sc = librosa.feature.spectral_centroid(y=x, sr=sr)[0]
    sb = librosa.feature.spectral_bandwidth(y=x, sr=sr)[0]
    features = []
    features.extend([np.mean(zcr),np.std(zcr),np.min(zcr),np.max(zcr)])
    features.extend([np.mean(sc),np.std(sc),np.min(sc),np.max(sc)])
    features.extend([np.mean(sb),np.std(sb),np.min(sb),np.max(sb)])
    for coeff in mfcc:
        features.extend([np.mean(coeff),np.std(coeff),np.min(coeff),np.max(coeff)])
    return np.array(features, dtype=np.float32)

In [ ]:
X, y = [], []
for idx, row in train_ds.iterrows():
    s1, sr = librosa.load(os.path.join(path, row["audio_path"]),sr=None)
    s2 = time_stretch(s1, rate=0.90)
    s3 = time_stretch(s1, rate=0.95)
    sp1 = time_stretch(s1, rate=1.20)
    sp2 = time_stretch(s1, rate=1.50)
    sp3 = librosa.resample(s1, orig_sr=sr, target_sr=8000)
    X.append(extract_features(s1, sr))
    y.append(0)
    for sig in [s2, s3, sp1, sp2, sp3]:
        X.append(extract_features(sig, sr))
        y.append(1)
X = np.array(X)
y = np.array(y)
print(X.shape, y.shape)

In [ ]:
X_test = []
for idx, row in test_ds.iterrows():
    x, sr= librosa.load(os.path.join(path, row["audio_path"]), sr = None)
    X_test.append(extract_features(x, sr))

In [ ]:
X_test= np.array(X_test)
X_test.shape

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
test_pred_cat = np.zeros(len(X_test))
test_pred_xgb = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    cat = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.01,
        depth=6,
        loss_function='Logloss',
        verbose=0,
        random_seed=42,
        task_type="GPU"
    )
    cat.fit(X_train, y_train)
    oof_cat[val_idx] = cat.predict_proba(X_val)[:, 1]
    test_pred_cat += cat.predict_proba(X_test)[:, 1] / skf.n_splits
    xgb = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        device="cuda"
    )
    xgb.fit(X_train, y_train)
    oof_xgb[val_idx] = xgb.predict_proba(X_val)[:, 1]
    test_pred_xgb += xgb.predict_proba(X_test)[:, 1] / skf.n_splits
    print(f"Fold {fold} done")
cat_score = roc_auc_score(y, oof_cat)
xgb_score = roc_auc_score(y, oof_xgb)
print("Cat OOF AUC:", cat_score)
print("XGB OOF AUC:", xgb_score)

In [ ]:
cat_w = cat_score / (cat_score + xgb_score)
xgb_w = xgb_score / (cat_score + xgb_score)
test_blend = cat_w * test_pred_cat + xgb_w * test_pred_xgb
sample_sub["score"] = test_blend

In [ ]:
sample_sub["score"]

In [ ]:
sample_sub.to_csv("sub5.csv" , index= False)